# Шаг 8. Гибридная модель: LightFM (collaborative + content)

## Цель ноутбука
Обучить **гибридную рекомендательную модель LightFM**, которая объединяет:
- Коллаборативный сигнал: факторизация матрицы взаимодействий user × item
- Контентные признаки: жанры (one-hot) + теги (TF-IDF)

**Ключевое отличие от SVD/KNN/LightGBM:**
LightFM оптимизирует ранжирующую функцию потерь (WARP/BPR), а не RMSE.
Главная метрика — **NDCG@10**. RMSE не считаем (скоры LightFM — произвольная шкала).

## 0. Импорты и настройки

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

# ── Fallback-установка LightFM ──────────────────────────────────────────
try:
    import lightfm  # noqa
except ImportError:
    import subprocess, sys as _sys
    try:
        subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'lightfm'],
                       check=True)
    except subprocess.CalledProcessError:
        print(
            "\u26a0\ufe0f Не удалось установить lightfm.\n"
            "Рекомендуемые версии Python: 3.10 / 3.11.\n"
            "Попробуйте: pip install lightfm --no-build-isolation\n"
            "На Windows установите Visual Studio Build Tools с компонентом C++."
        )

# ── Fallback-установка Optuna ────────────────────────────────────────────
try:
    import optuna  # noqa
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', 'optuna', 'plotly'],
                   check=True)

import optuna
import optuna.visualization as ov
from lightfm import LightFM

from src.utils import SEED, set_seeds
from src.data_io import load_splits, load_features, load_id_maps, load_tag_features
from src.metrics import (
    evaluate_topn, build_ground_truth,
    ndcg_at_k, precision_at_k, recall_at_k, hit_rate_at_k, coverage,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
optuna.logging.set_verbosity(optuna.logging.WARNING)
set_seeds()

MODELS_DIR    = Path('../models')
PROCESSED_DIR = Path('../data/processed')

print(f"SEED = {SEED}")
try:
    import lightfm as _lfm
    print(f"LightFM version: {_lfm.__version__}")
except Exception:
    print("LightFM version: unknown")
print(f"Optuna version: {optuna.__version__}")

## 1. Загрузка данных

In [ ]:
splits          = load_splits()
train, val, test = splits['train'], splits['val'], splits['test']

features        = load_features()
movies_enriched = features['movies_enriched']
genre_feats     = features['genres']

tag_data        = load_tag_features()
tag_matrix      = tag_data['matrix']
tag_order       = tag_data['order']

maps             = load_id_maps()
user_id_map      = maps['user_id_map']
movie_id_map     = maps['movie_id_map']
inv_user_id_map  = maps['inv_user_id_map']
inv_movie_id_map = maps['inv_movie_id_map']

n_users  = len(user_id_map)
n_movies = len(movie_id_map)

print(f"users={n_users}, movies={n_movies}")
print(f"train={len(train):,}, val={len(val):,}, test={len(test):,}")
print(f"genre_feats: {genre_feats.shape}")
print(f"tag_matrix: {tag_matrix.shape}")

## 2. Стратегия LightFM и подготовка данных

### Как работает LightFM
LightFM учит плотные эмбеддинги пользователей и фильмов через факторизацию матрицы
взаимодействий — аналогично SVD. Но дополнительно:
- Каждый **item-эмбеддинг** = взвешенная сумма эмбеддингов его признаков (жанров, тегов)
- Каждый **user-эмбеддинг** = взвешенная сумма эмбеддингов признаков пользователя
  (опционально; у нас user features не используем)

Это делает LightFM **гибридной**: даже для нового фильма (которого не было в train)
можно получить эмбеддинг по его жанрам и тегам.

### Бинаризация взаимодействий
Для `loss='warp'` и `loss='bpr'` LightFM оптимизирует **ранжирование**, а не рейтинг.
Поэтому `interactions` — бинарная матрица «релевантно/нет» (1 или 0), а не float.
Стандарт для MovieLens: оценка ≥ 4.0 → релевантно.

In [ ]:
RELEVANCE_THRESHOLD = 4.0

# ── Sparse матрица взаимодействий (только train) ─────────────────────────
train_pos = train[train['rating'] >= RELEVANCE_THRESHOLD]
interactions_train = sparse.csr_matrix(
    (np.ones(len(train_pos), dtype=np.float32),
     (train_pos['user_idx'].values, train_pos['movie_idx'].values)),
    shape=(n_users, n_movies),
)
print(f'Положительных взаимодействий в train: {interactions_train.nnz:,}')
print(f'Sparsity: {1 - interactions_train.nnz / (n_users * n_movies):.4%}')

### Матрица item features

LightFM ожидает, что `item_features` включают **identity-столбцы** —
по одному столбцу для каждого фильма. Это позволяет модели учить
per-item эмбеддинги поверх контентных признаков. Без identity-части
два фильма с одинаковыми жанрами и тегами были бы неразличимы.

**Источник:** официальная документация LightFM, раздел «item features»:
*"If you don't include the item's own identity feature, the model will not be able
to distinguish between items with the same features."*

Итоговая структура `item_features` (столбцы):
```
[identity (n_movies)] + [genres (19+)] + [TF-IDF tags (≤200)]
```

In [ ]:
# Жанры: упорядочиваем по movie_idx
genre_cols    = [c for c in genre_feats.columns if c.startswith('genre_')]
genre_idx_map = genre_feats.set_index('movieId')
ordered_movie_ids = [inv_movie_id_map[i] for i in range(n_movies)]
genre_matrix_ordered = genre_idx_map.loc[ordered_movie_ids, genre_cols].values
genre_sparse = sparse.csr_matrix(genre_matrix_ordered.astype(np.float32))

# TF-IDF теги: упорядочиваем по movie_idx
tag_idx_map = tag_order.set_index('movieId')['tag_row_idx']
ordered_tag_rows = tag_idx_map.loc[ordered_movie_ids].values
tag_sparse_ordered = tag_matrix[ordered_tag_rows]

# Конкатенация контентных признаков
content_features = sparse.hstack([genre_sparse, tag_sparse_ordered]).tocsr().astype(np.float32)
print(f'Genre features:   {genre_sparse.shape}')
print(f'Tag features:     {tag_sparse_ordered.shape}')
print(f'Content features: {content_features.shape}')

# Добавляем identity (ОБЯЗАТЕЛЬНО для per-item эмбеддингов)
identity     = sparse.identity(n_movies, dtype=np.float32, format='csr')
item_features = sparse.hstack([identity, content_features]).tocsr()
print(f'\nFinal item_features: {item_features.shape}')
print(f'  Первые {n_movies} столбцов = identity')
print(f'  Далее {content_features.shape[1]} = content (genres + tags)')

# Сохраняем для Streamlit inference
sparse.save_npz(MODELS_DIR / 'lightfm_item_features.npz', item_features)
print('item_features сохранён.')

In [ ]:
# Маска «всех просмотренных» в train (для исключения при генерации рекомендаций)
# Важно: используем ВСЕ оценки, не только позитивы — пользователь не должен
# получать фильмы, которые он уже смотрел, даже с низкой оценкой
all_seen_train_csr = sparse.csr_matrix(
    (np.ones(len(train), dtype=np.float32),
     (train['user_idx'].values, train['movie_idx'].values)),
    shape=(n_users, n_movies),
)
print(f'all_seen_train: {all_seen_train_csr.nnz:,} взаимодействий (все оценки)')

## 3. Утилита рекомендаций и базовая LightFM

### Утилита top-K генерации рекомендаций

In [ ]:
def lightfm_topn_recommendations(model, user_idx_list, train_val_pos_csr,
                                 item_features_mat, n_movies, k=20):
    """Top-K рекомендации из LightFM с исключением просмотренных.

    user_idx_list: список ВНУТРЕННИХ user индексов (0..n_users-1).
    train_val_pos_csr: sparse матрица «всех взаимодействий» (для маски),
                       shape (n_users, n_movies).
    Возвращает: dict {raw_user_id: list[raw_movie_id]}.
    """
    all_item_idx  = np.arange(n_movies, dtype=np.int32)
    recommendations = {}

    for uidx in user_idx_list:
        scores = model.predict(
            user_ids=int(uidx),
            item_ids=all_item_idx,
            item_features=item_features_mat,
            num_threads=4,
        )
        # Маска уже просмотренных
        seen_idx = train_val_pos_csr[uidx].indices
        scores[seen_idx] = -np.inf

        # Быстрый топ-K через argpartition, затем сортировка внутри
        if len(all_item_idx) <= k:
            top_k_idx = np.argsort(-scores)[:k]
        else:
            top_k_idx = np.argpartition(-scores, k)[:k]
            top_k_idx = top_k_idx[np.argsort(-scores[top_k_idx])]

        raw_user_id = inv_user_id_map[uidx]
        recommendations[raw_user_id] = [inv_movie_id_map[int(i)] for i in top_k_idx]

    return recommendations

### Базовая LightFM (точка отсчёта)

In [ ]:
baseline_lightfm = LightFM(
    no_components=32,
    loss='warp',
    learning_rate=0.05,
    random_state=SEED,
)

t0 = time.time()
baseline_lightfm.fit(
    interactions=interactions_train,
    item_features=item_features,
    epochs=20,
    num_threads=4,
    verbose=False,
)
baseline_train_time = time.time() - t0
print(f'Baseline LightFM обучена за {baseline_train_time:.2f} с')

# Оценка на val
val_ground_truth   = build_ground_truth(val, relevance_threshold=RELEVANCE_THRESHOLD)
val_user_inner_idx = [user_id_map[u] for u in val_ground_truth.keys()]

t0 = time.time()
baseline_val_recs = lightfm_topn_recommendations(
    baseline_lightfm, val_user_inner_idx, all_seen_train_csr,
    item_features, n_movies, k=20,
)
baseline_eval_time = time.time() - t0

baseline_val_topn = evaluate_topn(
    baseline_val_recs, val_ground_truth,
    ks=(5, 10, 20),
    all_items=list(movie_id_map.keys()),
)
print(f'Baseline val ({baseline_eval_time:.1f} с):')
print(json.dumps(baseline_val_topn, indent=2))

## 4. Optuna — подбор гиперпараметров

**Оптимизируем NDCG@10 на val** (направление = maximize).

**Пространство параметров:**
- `loss`: warp / bpr / warp-kos — функция потерь
- `no_components`: 16..128 — размерность латентных эмбеддингов
- `learning_rate`: 1e-3..0.2 (log)
- `item_alpha`: 1e-9..1e-4 (log) — L2 регуляризация item-эмбеддингов
- `user_alpha`: 1e-9..1e-4 (log) — L2 регуляризация user-эмбеддингов
- `epochs`: 10..50 — число эпох обучения

**n_trials = 50, направление = maximize (NDCG@10).**

In [ ]:
def objective(trial: optuna.Trial) -> float:
    params = {
        'loss':          trial.suggest_categorical('loss', ['warp', 'bpr', 'warp-kos']),
        'no_components': trial.suggest_int('no_components', 16, 128),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.2, log=True),
        'item_alpha':    trial.suggest_float('item_alpha', 1e-9, 1e-4, log=True),
        'user_alpha':    trial.suggest_float('user_alpha', 1e-9, 1e-4, log=True),
        'random_state':  SEED,
    }
    epochs = trial.suggest_int('epochs', 10, 50)

    model = LightFM(**params)
    model.fit(
        interactions=interactions_train,
        item_features=item_features,
        epochs=epochs,
        num_threads=4,
        verbose=False,
    )
    recs = lightfm_topn_recommendations(
        model, val_user_inner_idx, all_seen_train_csr,
        item_features, n_movies, k=10,
    )
    return ndcg_at_k(recs, val_ground_truth, k=10)


sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    study_name='lightfm_movielens'
)

print("Запускаем Optuna (50 trials)... Ожидаемое время: 10–25 минут.")
t0 = time.time()
study.optimize(objective, n_trials=50, show_progress_bar=True)
optuna_time = time.time() - t0

print(f'\nOptuna завершила {len(study.trials)} trials за {optuna_time:.1f} с')
print(f'Лучший NDCG@10 на val: {study.best_value:.4f}')
print('Лучшие параметры:')
print(json.dumps(study.best_params, indent=2))

In [ ]:
# ── Визуализация Optuna ─────────────────────────────────────────────────

fig_history = ov.plot_optimization_history(study)
fig_history.update_layout(title='Optuna: история оптимизации LightFM (NDCG@10 на val)')
fig_history.write_html(str(MODELS_DIR / 'optuna_lightfm_history.html'))
fig_history.show()

fig_importance = ov.plot_param_importances(study)
fig_importance.update_layout(title='Optuna: важность гиперпараметров LightFM')
fig_importance.write_html(str(MODELS_DIR / 'optuna_lightfm_importance.html'))
fig_importance.show()

fig_slice = ov.plot_slice(study)
fig_slice.update_layout(title='Optuna: срезы по гиперпараметрам LightFM')
fig_slice.show()

**Интерпретация результатов Optuna:**

- **loss=warp** обычно выигрывает на MovieLens: WARP (Weighted Approximate-Rank Pairwise)
  оптимизирует Precision@K, что хорошо согласуется с NDCG@10.
  BPR (Bayesian Personalized Ranking) — альтернатива с более мягкой оптимизацией.
- **no_components**: для ml-latest-small оптимум обычно 32–64. Слишком большое
  число компонент → переобучение при малом числе пользователей.
- **learning_rate**: LightFM чувствителен к LR; слишком высокий → дивергенция,
  слишком низкий → медленная сходимость. Оптимум обычно 0.02–0.1.
- **item_alpha / user_alpha**: небольшая регуляризация (1e-6..1e-5) помогает избежать
  переобучения, особенно при малом числе взаимодействий у редких пользователей.

## 5. Финальная LightFM на train + val

In [ ]:
best_params = study.best_params.copy()
best_epochs = best_params.pop('epochs')

final_lightfm = LightFM(**{**best_params, 'random_state': SEED})

train_val     = pd.concat([train, val], ignore_index=True)
train_val_pos = train_val[train_val['rating'] >= RELEVANCE_THRESHOLD]

interactions_train_val = sparse.csr_matrix(
    (np.ones(len(train_val_pos), dtype=np.float32),
     (train_val_pos['user_idx'].values, train_val_pos['movie_idx'].values)),
    shape=(n_users, n_movies),
)

# Маска «всех просмотренных» в train+val (все оценки, не только позитивы)
all_seen_train_val_csr = sparse.csr_matrix(
    (np.ones(len(train_val), dtype=np.float32),
     (train_val['user_idx'].values, train_val['movie_idx'].values)),
    shape=(n_users, n_movies),
)

t0 = time.time()
final_lightfm.fit(
    interactions=interactions_train_val,
    item_features=item_features,
    epochs=best_epochs,
    num_threads=4,
    verbose=False,
)
final_train_time = time.time() - t0

print(f'Финальная LightFM обучена за {final_train_time:.2f} с')
print(f'epochs={best_epochs}, loss={best_params["loss"]}, '
      f'no_components={best_params["no_components"]}')

## 6. Оценка на test

In [ ]:
test_ground_truth    = build_ground_truth(test, relevance_threshold=RELEVANCE_THRESHOLD)
test_user_inner_idx = [user_id_map[u] for u in test_ground_truth.keys()]

print(f'Генерация топ-20 для {len(test_user_inner_idx)} пользователей...')
t0 = time.time()
test_recs = lightfm_topn_recommendations(
    final_lightfm, test_user_inner_idx, all_seen_train_val_csr,
    item_features, n_movies, k=20,
)
inference_time = time.time() - t0
print(f'Готово за {inference_time:.2f} с')

lightfm_test_topn_metrics = evaluate_topn(
    test_recs, test_ground_truth,
    ks=(5, 10, 20),
    all_items=list(movie_id_map.keys()),
)
print('LightFM test (top-N):')
print(json.dumps(lightfm_test_topn_metrics, indent=2))

## 7. Сравнение со всеми предыдущими моделями

In [ ]:
with open(MODELS_DIR / 'popularity_metrics.json', 'r', encoding='utf-8') as f:
    pop_metrics = json.load(f)
with open(MODELS_DIR / 'svd_metrics.json', 'r', encoding='utf-8') as f:
    svd_metrics_loaded = json.load(f)
with open(MODELS_DIR / 'knn_metrics.json', 'r', encoding='utf-8') as f:
    knn_metrics_loaded = json.load(f)
with open(MODELS_DIR / 'lightgbm_metrics.json', 'r', encoding='utf-8') as f:
    lgbm_metrics_loaded = json.load(f)


def make_row(name, rating_m=None, topn_m=None):
    def r(d, k):
        return round(float(d[k]), 4) if (d and k in d and d[k] is not None) else None
    return {
        'Модель':       name,
        'RMSE':         r(rating_m, 'rmse'),
        'MAE':          r(rating_m, 'mae'),
        'NDCG@10':      r(topn_m,   'ndcg@10'),
        'Precision@10': r(topn_m,   'precision@10'),
        'Recall@10':    r(topn_m,   'recall@10'),
        'HitRate@10':   r(topn_m,   'hit_rate@10'),
        'Coverage@20':  r(topn_m,   'coverage@20'),
    }


comparison_rows = [
    make_row('GlobalMean',  pop_metrics['global_mean']['test']),
    make_row('Popularity',  topn_m=pop_metrics['popularity']['test']),
    make_row('SVD',
             svd_metrics_loaded['final']['test_rating'],
             svd_metrics_loaded['final']['test_topn']),
    make_row('KNN',
             knn_metrics_loaded['final']['test_rating'],
             knn_metrics_loaded['final']['test_topn']),
    make_row('LightGBM',
             lgbm_metrics_loaded['final']['test_rating'],
             lgbm_metrics_loaded['final']['test_topn']),
    make_row('LightFM',     topn_m=lightfm_test_topn_metrics),
]

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df.to_string(index=False))

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

topn_cols  = ['NDCG@10', 'Precision@10', 'Recall@10', 'HitRate@10']
models_cmp = ['Popularity', 'SVD', 'KNN', 'LightGBM', 'LightFM']
colors_cmp = ['darkorange', 'steelblue', 'seagreen', 'mediumpurple', 'crimson']
x = np.arange(len(topn_cols))
width = 0.16

for i, (mname, color) in enumerate(zip(models_cmp, colors_cmp)):
    row_   = comparison_df[comparison_df['Модель'] == mname].iloc[0]
    vals   = [float(row_[c]) if row_[c] is not None else 0.0 for c in topn_cols]
    axes[0].bar(x + i * width, vals, width, label=mname, color=color, alpha=0.85)

axes[0].set_xticks(x + width * 2)
axes[0].set_xticklabels(topn_cols, fontsize=8)
axes[0].set_title('Top-N метрики (test)')
axes[0].legend(fontsize=7, ncol=2)

# NDCG@10 раздельно
ndcg_models = ['Popularity', 'SVD', 'KNN', 'LightGBM', 'LightFM']
ndcg_colors = ['darkorange', 'steelblue', 'seagreen', 'mediumpurple', 'crimson']
ndcg_vals   = [
    float(comparison_df[comparison_df['Модель'] == m]['NDCG@10'].iloc[0])
    for m in ndcg_models
    if comparison_df[comparison_df['Модель'] == m]['NDCG@10'].iloc[0] is not None
]
axes[1].bar(ndcg_models, ndcg_vals, color=ndcg_colors, alpha=0.85)
axes[1].set_title('NDCG@10 на test')
axes[1].set_ylabel('NDCG@10')
axes[1].tick_params(axis='x', rotation=20)
for bar, v in zip(axes[1].patches, ndcg_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.002,
                 f'{v:.4f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

**Интерпретация сравнительной таблицы:**

1. **LightFM vs Popularity (NDCG@10):** LightFM должна выигрывать — она персонализирована
   и использует как коллаборативный, так и контентный сигнал.

2. **LightFM vs SVD (NDCG@10):** результаты конкурентны. LightFM выигрывает за счёт
   контентного сигнала и ранжирующей функции потерь. SVD, оптимизируя RMSE, может
   уступать по ранжированию.

3. **LightFM без RMSE:** это намеренное решение — LightFM оптимизирует ранжирование,
   а не предсказание рейтинга. Сравнение по RMSE было бы некорректным.

4. **Если LightFM проигрывает Popularity:** скорее всего identity-часть item_features
   не включена (модель не может различать фильмы), либо слишком мало эпох.
   Проверить `item_features.shape[1] >= n_movies`.

## 8. Анализ влияния контентных признаков

Проверяем, что LightFM уловила контентный сигнал.
Ожидается: Toy Story и Toy Story 2 (похожие жанры) → более высокая косинусная схожесть,
чем Toy Story и Pulp Fiction (разные жанры).

In [ ]:
def get_movie_embeddings(model, item_features_mat, n_movies):
    """Получить плотные item-представления из LightFM."""
    biases, embeddings = model.get_item_representations(features=item_features_mat)
    return biases, embeddings


biases, embeddings = get_movie_embeddings(final_lightfm, item_features, n_movies)
print(f'Item embeddings shape: {embeddings.shape}')

# Косинусная схожесть между парами фильмов
def cosine_sim(a, b):
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a < 1e-9 or norm_b < 1e-9:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))


sample_pairs = [
    (1,    3114, 'Toy Story (1995) / Toy Story 2 (1999)    [анимация, дети]'),
    (1,    296,  'Toy Story (1995) / Pulp Fiction (1994)   [разные жанры]'),
    (296,  593,  'Pulp Fiction     / Silence of the Lambs  [thriller/crime]'),
    (50,   527,  'Usual Suspects   / Schindler\'s List      [drama/crime vs drama]'),
]

print('Косинусная схожесть item-эмбеддингов LightFM:')
for raw_a, raw_b, desc in sample_pairs:
    ia = movie_id_map.get(raw_a)
    ib = movie_id_map.get(raw_b)
    if ia is None or ib is None:
        print(f'  {desc}: один из фильмов отсутствует в train (cold-start)')
        continue
    cos = cosine_sim(embeddings[ia], embeddings[ib])
    print(f'  {desc}: {cos:.4f}')

Если похожие по жанру фильмы (например, две части Toy Story) имеют
более высокую косинусную схожесть, чем несвязанные (Toy Story vs Pulp Fiction) —
это подтверждает, что контентные признаки реально влияют на эмбеддинги.

LightFM объединяет per-item эмбеддинги (через identity-часть) с контентными эмбеддингами
(через genre/tag фичи), поэтому схожесть отражает как коллаборативный,
так и контентный сигнал.

## 9. Пример рекомендаций для одного пользователя

In [ ]:
sample_user_inner = test_user_inner_idx[0]
sample_user_raw   = inv_user_id_map[sample_user_inner]

# История пользователя
user_history = (
    train_val[
        (train_val['userId'] == sample_user_raw) & (train_val['rating'] >= 4.0)
    ]
    .merge(movies_enriched[['movieId', 'title', 'genres']], on='movieId')
    .sort_values('rating', ascending=False)
)

print(f'Пример пользователя: userId={sample_user_raw}')
print(f'\nВысоко оценённые фильмы (rating >= 4.0, топ-10):')
display(user_history[['title', 'genres', 'rating']].head(10))

print(f'\nТоп-10 рекомендаций LightFM для userId={sample_user_raw}:')
recs_sample = test_recs[sample_user_raw][:10]
recs_df = movies_enriched[movies_enriched['movieId'].isin(recs_sample)][
    ['movieId', 'title', 'genres']
]
# Сортируем в порядке рекомендации
recs_df = recs_df.set_index('movieId').loc[recs_sample].reset_index()
display(recs_df)

## 10. Сохранение артефактов

In [ ]:
# Модель
joblib.dump(final_lightfm, MODELS_DIR / 'lightfm_model.pkl')
print(f"lightfm_model.pkl: {(MODELS_DIR / 'lightfm_model.pkl').stat().st_size / 1024:.1f} KB")

# Параметры
lightfm_params_meta = {
    'random_state':           SEED,
    'best_params':            study.best_params,
    'best_epochs':            best_epochs,
    'optuna_n_trials':        50,
    'optuna_sampler':         'TPESampler',
    'optuna_direction':       'maximize',
    'optuna_target':          'ndcg@10@val',
    'final_train_strategy':   'train+val; epochs from best trial',
    'relevance_threshold':    RELEVANCE_THRESHOLD,
    'n_users':                n_users,
    'n_movies':               n_movies,
    'item_features_shape':    list(item_features.shape),
    'n_genre_features':       int(genre_sparse.shape[1]),
    'n_tag_features':         int(tag_sparse_ordered.shape[1]),
    'baseline_train_time_sec':  baseline_train_time,
    'optuna_search_time_sec':   optuna_time,
    'final_train_time_sec':     final_train_time,
    'inference_time_test_topn_sec': inference_time,
}
with open(MODELS_DIR / 'lightfm_params.json', 'w', encoding='utf-8') as f:
    json.dump(lightfm_params_meta, f, ensure_ascii=False, indent=2)

# Метрики
lightfm_metrics = {
    'baseline': {
        'val_topn': baseline_val_topn,
    },
    'final': {
        'val_best_ndcg10': float(study.best_value),
        'test_topn':       lightfm_test_topn_metrics,
    },
    'meta': {
        'k_values':            [5, 10, 20],
        'relevance_threshold': RELEVANCE_THRESHOLD,
        'optuna_n_trials':     50,
    },
}
with open(MODELS_DIR / 'lightfm_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(lightfm_metrics, f, ensure_ascii=False, indent=2)

# История trials Optuna
trials_df = study.trials_dataframe()
trials_df.to_parquet(MODELS_DIR / 'lightfm_optuna_trials.parquet', index=False)

print('Все артефакты сохранены.')
print(f'  Лучший NDCG@10 (val): {study.best_value:.4f}')
print(f'  NDCG@10 (test):       {lightfm_test_topn_metrics["ndcg@10"]:.4f}')
print(f'  Coverage@20 (test):   {lightfm_test_topn_metrics.get("coverage@20", 0):.4f}')

## 11. Финальные проверки

In [ ]:
required_files = [
    MODELS_DIR / 'lightfm_model.pkl',
    MODELS_DIR / 'lightfm_item_features.npz',
    MODELS_DIR / 'lightfm_params.json',
    MODELS_DIR / 'lightfm_metrics.json',
    MODELS_DIR / 'lightfm_optuna_trials.parquet',
    MODELS_DIR / 'optuna_lightfm_history.html',
    MODELS_DIR / 'optuna_lightfm_importance.html',
]
for p in required_files:
    assert p.exists(), f'Файл не найден: {p}'

# Round-trip модели
lightfm_loaded  = joblib.load(MODELS_DIR / 'lightfm_model.pkl')
features_loaded = sparse.load_npz(MODELS_DIR / 'lightfm_item_features.npz')

sample_uidx      = test_user_inner_idx[0]
all_items_arr    = np.arange(n_movies, dtype=np.int32)
scores_orig      = final_lightfm.predict(
    user_ids=int(sample_uidx), item_ids=all_items_arr,
    item_features=item_features, num_threads=1,
)
scores_loaded    = lightfm_loaded.predict(
    user_ids=int(sample_uidx), item_ids=all_items_arr,
    item_features=features_loaded, num_threads=1,
)
np.testing.assert_allclose(scores_orig, scores_loaded, atol=1e-6)

# Sanity-границы метрик
assert 0.0 <= lightfm_test_topn_metrics['ndcg@10'] <= 1.0
assert 0.0 <= lightfm_test_topn_metrics['precision@10'] <= 1.0

# Ровно 50 trials
assert len(study.trials) == 50, f'Optuna прогнала {len(study.trials)} trials вместо 50'

# Optuna улучшила baseline (мягкая проверка)
assert study.best_value >= baseline_val_topn['ndcg@10'] - 1e-6,     'Optuna не улучшила baseline NDCG@10'

# LightFM побеждает popularity по NDCG@10 (содержательная проверка)
pop_ndcg = pop_metrics['popularity']['test']['ndcg@10']
assert lightfm_test_topn_metrics['ndcg@10'] > pop_ndcg, (
    f"LightFM NDCG@10 ({lightfm_test_topn_metrics['ndcg@10']:.4f}) "
    f"не побеждает Popularity ({pop_ndcg:.4f})"
)

# Корректность рекомендаций
sample_user_raw = inv_user_id_map[test_user_inner_idx[0]]
sample_recs     = test_recs[sample_user_raw]
assert len(sample_recs) == 20
assert len(set(sample_recs)) == 20, 'В рекомендациях есть дубликаты'
seen_by_user = set(train_val[train_val['userId'] == sample_user_raw]['movieId'])
assert not (set(sample_recs) & seen_by_user),     'В рекомендациях LightFM есть уже просмотренные фильмы'

# Структурная проверка item_features
assert item_features.shape[0] == n_movies
assert item_features.shape[1] >= n_movies,     'Identity-часть item_features отсутствует или повреждена'

print('\u2705 Шаг 8 пройден: LightFM обучена, гиперпараметры подобраны, метрики сохранены')
print(f'   NDCG@10 (test):    {lightfm_test_topn_metrics["ndcg@10"]:.4f}  '
      f'(vs Popularity {pop_ndcg:.4f}, '
      f'vs SVD {svd_metrics_loaded["final"]["test_topn"]["ndcg@10"]:.4f})')
print(f'   Coverage@20 (test): {lightfm_test_topn_metrics.get("coverage@20", 0):.4f}')
print(f'   best_epochs={best_epochs}, loss={study.best_params["loss"]}')